# How Credit Shapes Economic Growth: Insights from World Bank Data

## Notebook Roadmap

This notebook follows the **CRISP-DM** (Cross-Industry Standard Process for Data Mining) methodology:

1. **Business Understanding** - Define objectives and research questions
2. **Data Understanding** - Gather and explore the dataset
3. **Data Preparation** - Clean and prepare data for analysis
4. **Modeling** - Build predictive models
5. **Evaluation** - Assess model performance
6. **Deployment/Conclusions** - Summarize findings and implications

---

# 1. Business Understanding

## Project Overview

Economic growth is a key indicator of a country's prosperity and is influenced by many factors. Understanding the relationship between credit availability and GDP growth can help policymakers design effective economic policies.

## Questions We Aim to Answer

This analysis addresses the following business/real-world questions:

1. **Question 1:** Which countries have the highest GDP growth, and what patterns do we observe?
2. **Question 2:** How does credit availability correlate with GDP growth across countries?
3. **Question 3:** What is the relationship between inflation and unemployment (Phillips Curve)?
4. **Question 4:** Can we predict GDP growth based on credit and other economic indicators?

## Why This Matters

Understanding these relationships helps:
- Policymakers make informed decisions about credit policies
- Economists understand economic growth drivers
- Investors assess economic conditions in different countries

# 2. Data Understanding (Gather & Assess)

## Data Source

The data is sourced from the [World Bank Databank](https://databank.worldbank.org/), which provides comprehensive economic indicators for countries worldwide.

For demonstration purposes, we generate synthetic data that mimics the structure of real World Bank data. This approach allows the notebook to run without requiring external data downloads.

In [ ]:
# Import required libraries
# We use pandas for data manipulation, numpy for numerical operations,
# matplotlib and seaborn for visualization, and sklearn for modeling

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import os
import sys

# Import our utility functions from src module
# These functions contain docstrings and are reusable for similar analyses
sys.path.insert(0, 'src')
from utils import (
    calculate_model_metrics,
    clean_dataframe,
    generate_synthetic_economic_data,
    predict_gdp_growth
)

# Set visualization style for consistent, publication-ready plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Create images folder for saving visualizations
os.makedirs('images', exist_ok=True)

print("Libraries loaded successfully!")

## 2.1 Data Gathering

We generate synthetic economic data with the following indicators:
- **Credit_to_private_sector**: Domestic credit to private sector (% of GDP)
- **Inflation**: Consumer price inflation, annual (%)
- **Unemployment**: Unemployment rate, total (% of labor force)
- **GDP_growth**: GDP growth rate, annual (%)

In [ ]:
# Generate synthetic data using our utility function
# We use 200 samples to simulate data from 200 countries
# Setting random_seed=42 ensures reproducibility of results

data = generate_synthetic_economic_data(n_samples=200, random_seed=42)

# Display first few rows to understand data structure
print(f"Dataset shape: {data.shape}")
print(f"\nFirst 5 rows of the dataset:")
data.head()

## 2.2 Data Assessment

Let's examine the dataset to understand its characteristics, check for missing values, and compute summary statistics.

In [ ]:
# Check data types and memory usage
# This helps us understand if any type conversion is needed
print("Data types and info:")
data.info()

In [ ]:
# Generate summary statistics for numeric columns
# This shows distribution characteristics: min, max, mean, std, quartiles
print("\nSummary statistics:")
data.describe()

In [ ]:
# Check for missing values
# Missing data can affect our analysis and modeling
print("\nMissing values per column:")
print(data.isnull().sum())
print(f"\nTotal missing values: {data.isnull().sum().sum()}")

# 3. Data Preparation (Clean)

## Why Data Cleaning is Important

Before analysis and modeling, we need to ensure our data is clean and ready:
- Remove or handle missing values
- Check for and handle outliers if necessary
- Ensure correct data types

In [ ]:
# Clean the data using our utility function
# This function handles missing values based on the drop_na parameter
# We drop rows with any missing values to ensure complete cases for analysis

data_clean = clean_dataframe(data, drop_na=True)

print(f"Original dataset size: {len(data)} rows")
print(f"Cleaned dataset size: {len(data_clean)} rows")
print(f"Rows removed: {len(data) - len(data_clean)}")

---

# 4. Questions and Analysis

Now we address each of our business questions with visualizations and statistical analysis.

## Question 1: Which countries have the highest GDP growth?

### Approach
We'll visualize the distribution of GDP growth across all countries to understand:
- The range of GDP growth values
- The central tendency (where most countries cluster)
- Any outliers or unusual patterns

### Why a Histogram with KDE?
A histogram shows the frequency distribution, while the kernel density estimate (KDE) overlay provides a smooth representation of the distribution shape.

In [ ]:
# Question 1: Analyze GDP Growth Distribution
# Create a histogram with kernel density estimate to visualize the distribution

plt.figure(figsize=(10, 6))
sns.histplot(data_clean['GDP_growth'], bins=30, kde=True, color='steelblue')
plt.title('Distribution of GDP Growth Across Countries', fontsize=14)
plt.xlabel('GDP Growth (%)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)

# Add mean line for reference
mean_gdp = data_clean['GDP_growth'].mean()
plt.axvline(mean_gdp, color='red', linestyle='--', label=f'Mean: {mean_gdp:.2f}%')
plt.legend()

plt.tight_layout()
plt.savefig('images/gdp_growth_distribution.png', dpi=150)
plt.show()

# Display top 5 countries by GDP growth
print("\nTop 5 Countries by GDP Growth:")
top_countries = data_clean.nlargest(5, 'GDP_growth')[['Country', 'GDP_growth']]
print(top_countries.to_string(index=False))

### Conclusion for Question 1

**Finding:** GDP growth is approximately uniformly distributed across countries in this synthetic dataset, ranging from about -5% to +10%. The top-performing countries show GDP growth rates near the upper bound of 10%.

**Implication:** In real-world data, we would expect to see more clustering around typical growth rates (2-4%) with fewer extreme values. The uniform distribution here is an artifact of synthetic data generation.

## Question 2: How does credit availability correlate with GDP growth?

### Approach
We'll create a correlation heatmap to examine relationships between all economic indicators, with special attention to the credit-GDP relationship.

### Why a Correlation Heatmap?
A heatmap provides an intuitive visual representation of the correlation matrix, allowing us to quickly identify strong positive (red) or negative (blue) relationships between variables.

In [ ]:
# Question 2: Analyze correlations between economic indicators
# Select only numeric columns for correlation analysis

numeric_cols = ['Credit_to_private_sector', 'Inflation', 'Unemployment', 'GDP_growth']
correlation_matrix = data_clean[numeric_cols].corr()

# Create a correlation heatmap with annotations
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, 
            annot=True, 
            cmap='coolwarm', 
            center=0,
            square=True,
            linewidths=0.5,
            fmt='.3f')
plt.title('Correlation Heatmap: Economic Indicators', fontsize=14)
plt.tight_layout()
plt.savefig('images/correlation_heatmap.png', dpi=150)
plt.show()

# Print specific correlation value
credit_gdp_corr = correlation_matrix.loc['Credit_to_private_sector', 'GDP_growth']
print(f"\nCorrelation between Credit to Private Sector and GDP Growth: {credit_gdp_corr:.4f}")

In [ ]:
# Create scatter plot to visualize Credit vs GDP Growth relationship
# This helps us see the actual data points and any patterns

plt.figure(figsize=(10, 6))
sns.regplot(x='Credit_to_private_sector', y='GDP_growth', data=data_clean, 
            scatter_kws={'alpha': 0.5}, line_kws={'color': 'red'})
plt.title('Credit to Private Sector vs GDP Growth', fontsize=14)
plt.xlabel('Credit to Private Sector (% of GDP)', fontsize=12)
plt.ylabel('GDP Growth (%)', fontsize=12)
plt.tight_layout()
plt.savefig('images/credit_vs_gdp.png', dpi=150)
plt.show()

### Conclusion for Question 2

**Finding:** The correlation between credit availability and GDP growth is very weak (close to 0) in this synthetic dataset. This is expected because the data was generated randomly without any built-in relationships.

**Implication:** In real-world data from the World Bank, we would typically expect to find a moderate positive correlation, as countries with better access to credit tend to have more investment and economic activity.

## Question 3: What is the relationship between inflation and unemployment?

### Approach
We examine the Phillips Curve relationship - the historically observed inverse relationship between inflation and unemployment.

### Why This Matters
The Phillips Curve is a fundamental concept in macroeconomics. Understanding this relationship helps policymakers balance inflation control with unemployment reduction.

In [ ]:
# Question 3: Analyze Inflation vs Unemployment (Phillips Curve)
# Create scatter plot with regression line

plt.figure(figsize=(10, 6))
sns.regplot(x='Unemployment', y='Inflation', data=data_clean,
            scatter_kws={'alpha': 0.5}, line_kws={'color': 'red'})
plt.title('Inflation vs Unemployment (Phillips Curve Analysis)', fontsize=14)
plt.xlabel('Unemployment Rate (%)', fontsize=12)
plt.ylabel('Inflation Rate (%)', fontsize=12)
plt.tight_layout()
plt.savefig('images/phillips_curve.png', dpi=150)
plt.show()

# Calculate and display correlation
inflation_unemployment_corr = data_clean['Inflation'].corr(data_clean['Unemployment'])
print(f"Correlation between Inflation and Unemployment: {inflation_unemployment_corr:.4f}")

### Conclusion for Question 3

**Finding:** The correlation between inflation and unemployment is weak in this synthetic dataset. There is no clear inverse relationship (Phillips Curve) visible.

**Implication:** In real-world data, the Phillips Curve relationship varies by country and time period. Modern economists recognize that the relationship is not always stable and can shift based on expectations and policy.

# 5. Modeling

## Question 4: Can we predict GDP growth based on economic indicators?

### Approach
We build a Linear Regression model to predict GDP growth using:
- Credit to private sector
- Inflation
- Unemployment

### Why Linear Regression?
Linear regression is chosen because:
1. It provides interpretable coefficients showing the relationship between each predictor and GDP growth
2. It's a good baseline model for understanding variable relationships
3. The results are easy to communicate to stakeholders

In [ ]:
# Prepare features (X) and target variable (y)
# Features are the economic indicators; target is GDP growth

feature_cols = ['Credit_to_private_sector', 'Inflation', 'Unemployment']
X = data_clean[feature_cols]
y = data_clean['GDP_growth']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# Split data into training and testing sets
# We use 80% for training and 20% for testing
# random_state=42 ensures reproducibility

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")

In [ ]:
# Train the Linear Regression model
# The model learns the relationship between features and GDP growth

model = LinearRegression()
model.fit(X_train, y_train)

# Display model coefficients for interpretation
print("Model Coefficients:")
for feature, coef in zip(feature_cols, model.coef_):
    print(f"  {feature}: {coef:.4f}")
print(f"  Intercept: {model.intercept_:.4f}")

In [ ]:
# Make predictions on the test set
y_pred = model.predict(X_test)

# 6. Evaluation

## Model Performance Assessment

We evaluate the model using:
- **R² (R-squared)**: Proportion of variance explained by the model (0 to 1, higher is better)
- **RMSE (Root Mean Squared Error)**: Average prediction error in the same units as GDP growth (%)

### Why These Metrics?
- R² tells us how well the model explains the variability in GDP growth
- RMSE gives us the typical prediction error magnitude in interpretable units

In [ ]:
# Calculate model performance metrics using our utility function
# This provides a standardized way to evaluate regression models

metrics = calculate_model_metrics(y_test, y_pred)

print("Model Performance:")
print(f"  R² Score: {metrics['r2']:.4f}")
print(f"  RMSE: {metrics['rmse']:.4f}%")

In [ ]:
# Visualize actual vs predicted values
# Points close to the diagonal line indicate good predictions

plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual GDP Growth (%)', fontsize=12)
plt.ylabel('Predicted GDP Growth (%)', fontsize=12)
plt.title('Actual vs Predicted GDP Growth', fontsize=14)
plt.tight_layout()
plt.savefig('images/actual_vs_predicted.png', dpi=150)
plt.show()

### Model Interpretation

The low R² score (~0.02) indicates that the model explains very little of the variance in GDP growth. This is expected because:
1. The data is synthetically generated without real relationships
2. In reality, GDP growth depends on many more factors than just three indicators

## Scenario Prediction

Let's use the model to predict GDP growth for a hypothetical country scenario.

In [ ]:
# Scenario prediction using our utility function
# Hypothetical country with moderate credit, low inflation, low unemployment

predicted_growth = predict_gdp_growth(
    model,
    credit=50,      # 50% of GDP
    inflation=3,     # 3% inflation
    unemployment=5   # 5% unemployment
)

print("Scenario Prediction:")
print(f"  Credit to Private Sector: 50% of GDP")
print(f"  Inflation: 3%")
print(f"  Unemployment: 5%")
print(f"  \n  Predicted GDP Growth: {predicted_growth:.2f}%")

### Conclusion for Question 4

**Finding:** The linear regression model shows very low predictive power (R² ≈ 0.02) for GDP growth based on the three economic indicators.

**Implication:** This suggests that either:
1. The relationship between these variables and GDP growth is non-linear
2. Other factors not included in the model are more important drivers of GDP growth
3. With real-world data showing actual relationships, the model would likely perform better

# 7. Deployment/Conclusions

## Summary of Findings

| Question | Key Finding |
|----------|-------------|
| Q1: GDP Growth Distribution | GDP growth ranges from -5% to +10%, with uniform distribution in synthetic data |
| Q2: Credit vs GDP | Weak correlation between credit availability and GDP growth |
| Q3: Inflation vs Unemployment | No clear Phillips Curve relationship observed |
| Q4: Predictive Model | Linear regression has low predictive power (R² ≈ 0.02) |

## Key Takeaways

1. **Credit availability matters, but isn't everything**: While credit to private sector is often associated with economic growth, our analysis shows it's just one of many factors.

2. **Economic relationships are complex**: The weak correlations observed suggest that simple linear relationships don't fully capture economic dynamics.

3. **Data quality is crucial**: Using synthetic data limited our ability to draw real conclusions. Real World Bank data would provide more meaningful insights.

4. **Model limitations**: Linear regression is a good starting point but may not capture non-linear relationships in economic data.

## Recommendations for Future Work

1. Use real World Bank data for more meaningful analysis
2. Include additional economic indicators (trade balance, government spending, etc.)
3. Try non-linear models (Random Forest, Gradient Boosting)
4. Conduct time-series analysis to capture temporal patterns
5. Consider regional and development-level groupings

## Acknowledgements

- Data concept from [World Bank Databank](https://databank.worldbank.org/)
- This analysis was completed as part of the Udacity Data Scientist Nanodegree Program